# Option 1: One skill per employee - one skill per task

## 1. Problem Definition
In this initial phase, we are solving a static, continuous resource allocation problem. Our goal is to assign employee hours to specific project tasks over a 1-year period to minimize total payroll costs. 

To ensure realism, an employee can only be assigned to a task if their `Primary_Skill` matches the task's `Required_Skill`. 

## 2. Mathematical Formulation
Let us define the components of our Linear Programming (LP) model:

**Sets:**
* $E$: Set of all employees, indexed by $i$.
* $T$: Set of all project tasks, indexed by $j$.
* $V$: Set of all valid assignment pairs $(i,j)$ where employee $i$'s skill matches task $j$'s required skill.

**Parameters:**
* $c_i$: Hourly cost of employee $i$.
* $k_i$: Maximum annual available hours for employee $i$ (e.g., 2080).
* $d_j$: Total hours demanded by task $j$.

**Decision Variables:**
* $x_{ij}$: The continuous number of hours employee $i$ is assigned to task $j$. Defined only for $(i,j) \in V$.

**Objective Function:**
Minimize the total cost of task assignments:
$$\min Z = \sum_{(i,j) \in V} c_i \cdot x_{ij}$$

**Constraints:**
1. **Demand Satisfaction:** Every task must receive exactly the hours it requires.
$$\sum_{i \mid (i,j) \in V} x_{ij} = d_j \quad \forall j \in T$$

2. **Capacity Limit:** No employee can exceed their maximum available hours.
$$\sum_{j \mid (i,j) \in V} x_{ij} \le k_i \quad \forall i \in E$$

3. **Non-Negativity:** Hours assigned cannot be negative.
$$x_{ij} \ge 0 \quad \forall (i,j) \in V$$

In [ ]:
# Install pulp if you haven't already: !pip install pulp
import pandas as pd
import numpy as np
import random

## 3. Synthetic Data Generation
Instead of relying on static CSVs, we generate our own realistic dataset. This allows us to easily scale the problem size in future phases to test the solver's performance.

We will generate:
1. **10 Employees** with random skills (Backend, Frontend, QA), costs, and standard 2080-hour capacities.
2. **5 Projects** broken down into specific skill-based tasks.

In [9]:
# Set seed for reproducibility
# random.seed(42)
# np.random.seed(42)

skills = ['Backend', 'Frontend', 'QA']

# 1. Generate Employees
employee_data = []
for i in range(1, 11):
    emp_id = f"E{i:02d}"
    skill = random.choice(skills)
    # Base cost depends on skill to add variation
    base_cost = 70 if skill == 'Backend' else (60 if skill == 'Frontend' else 40)
    cost = base_cost + random.randint(-10, 20) 
    
    employee_data.append({
        'Employee_ID': emp_id,
        'Primary_Skill': skill,
        'Hourly_Cost': cost,
        'Max_Hours': 2080
    })

employees_df = pd.DataFrame(employee_data)

# 2. Generate Tasks (Demand)
task_data = []
project_count = 5

for p in range(1, project_count + 1):
    proj_id = f"P{p:02d}"
    # Randomly decide which roles this project needs
    roles_needed = random.sample(skills, k=random.randint(2, 3)) 
    
    for role in roles_needed:
        task_id = f"{proj_id}_{role}"
        # Generate between 200 and 1000 hours needed per task
        hours = random.randint(2, 10) * 100 
        
        task_data.append({
            'Task_ID': task_id,
            'Project_ID': proj_id,
            'Required_Skill': role,
            'Hours_Needed': hours
        })

tasks_df = pd.DataFrame(task_data)

print("--- Employees ---")
display(employees_df.head(5))
print("\n--- Tasks ---")
display(tasks_df.head(5))

--- Employees ---


,Employee_ID,Primary_Skill,Hourly_Cost,Max_Hours
0,E01,QA,49,2080
1,E02,Frontend,68,2080
2,E03,Backend,82,2080
3,E04,Backend,61,2080
4,E05,QA,37,2080



--- Tasks ---


,Task_ID,Project_ID,Required_Skill,Hours_Needed
0,P01_Backend,P01,Backend,500
1,P01_Frontend,P01,Frontend,600
2,P01_QA,P01,QA,300
3,P02_QA,P02,QA,400
4,P02_Backend,P02,Backend,900


## 4. Building and Solving the Model with PuLP
Here, we translate our mathematical equations into Python using the `pulp` library. 

A critical step is filtering for **Valid Pairs**. We only create a decision variable $x_{ij}$ if the employee possesses the skill required for the task. This drastically reduces the computational size of the model and guarantees no unqualified assignments.

In [11]:
from ortools.linear_solver import pywraplp

# Initialize the OR-Tools solver using GLOP (Google's Linear Programming solver)
solver = pywraplp.Solver.CreateSolver('GLOP')

# Create a list of Valid Pairs (i, j) based on skill matching
valid_pairs = []
for _, emp in employees_df.iterrows():
    for _, task in tasks_df.iterrows():
        if emp['Primary_Skill'] == task['Required_Skill']:
            valid_pairs.append((emp['Employee_ID'], task['Task_ID']))

# Define Decision Variables: x_ij (continuous, lower bound = 0)
x = {}
for (i, j) in valid_pairs:
    x[(i, j)] = solver.NumVar(0, solver.infinity(), f'x_{i}_{j}')

# Define Objective Function: Minimize Cost (Hours * Hourly_Cost)
cost_dict = employees_df.set_index('Employee_ID')['Hourly_Cost'].to_dict()
objective = solver.Objective()
for (i, j) in valid_pairs:
    objective.SetCoefficient(x[(i, j)], float(cost_dict[i]))
objective.SetMinimization()

# Constraint 1: Demand Satisfaction (Task must get exact hours needed)
demand_dict = tasks_df.set_index('Task_ID')['Hours_Needed'].to_dict()
for j in tasks_df['Task_ID']:
    capable_employees = [i for (i, t_id) in valid_pairs if t_id == j]
    constraint = solver.Constraint(float(demand_dict[j]), float(demand_dict[j]), f"Demand_{j}")
    for i in capable_employees:
        constraint.SetCoefficient(x[(i, j)], 1)

# Constraint 2: Capacity Limit (Employee cannot exceed Max_Hours)
cap_dict = employees_df.set_index('Employee_ID')['Max_Hours'].to_dict()
for i in employees_df['Employee_ID']:
    assigned_tasks = [j for (e_id, j) in valid_pairs if e_id == i]
    # Lower bound 0, upper bound is their max capacity
    constraint = solver.Constraint(0, float(cap_dict[i]), f"Capacity_{i}")
    for j in assigned_tasks:
        constraint.SetCoefficient(x[(i, j)], 1)

# Solve the model
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Status: OPTIMAL")
    print(f"Total Optimized Cost: ${solver.Objective().Value():,.2f}")
else:
    print("The problem does not have an optimal solution.")

Status: OPTIMAL
Total Optimized Cost: $358,720.00


In [10]:
# Calculate Total Supply by Skill
supply = employees_df.groupby('Primary_Skill')['Max_Hours'].sum()

# Calculate Total Demand by Skill
demand = tasks_df.groupby('Required_Skill')['Hours_Needed'].sum()

# Combine and Compare
capacity_check = pd.DataFrame({'Total_Supply_Hours': supply, 'Total_Demand_Hours': demand}).fillna(0)
capacity_check['Difference (Supply - Demand)'] = capacity_check['Total_Supply_Hours'] - capacity_check['Total_Demand_Hours']

print("--- Supply vs. Demand Check ---")
display(capacity_check)

--- Supply vs. Demand Check ---


,Total_Supply_Hours,Total_Demand_Hours,Difference (Supply - Demand)
Backend,8320,2900,5420
Frontend,6240,1900,4340
QA,6240,2000,4240


## 5. Results Extraction
The solver has found the optimal allocation. We will now extract the non-zero variables to see exactly who is working on what, and verify that the math constraints held up.

In [12]:
# Extract the results into a readable format
results = []
for (i, j) in valid_pairs:
    assigned_hours = x[(i, j)].solution_value()
    if assigned_hours > 0:  # Only show actual assignments
        cost = assigned_hours * cost_dict[i]
        results.append({
            'Employee_ID': i,
            'Task_ID': j,
            'Assigned_Hours': assigned_hours,
            'Cost_Generated': cost
        })

results_df = pd.DataFrame(results)

print("--- Optimal Assignment Schedule ---")
display(results_df.sort_values(by=['Task_ID', 'Employee_ID']).reset_index(drop=True))

# Sanity Check: Did any employee exceed 2080 hours?
print("\n--- Employee Utilization Check ---")
utilization = results_df.groupby('Employee_ID')['Assigned_Hours'].sum().reset_index()
utilization = utilization.merge(employees_df[['Employee_ID', 'Max_Hours']], on='Employee_ID')
utilization['Utilization_%'] = (utilization['Assigned_Hours'] / utilization['Max_Hours'] * 100).round(1)
display(utilization)

--- Optimal Assignment Schedule ---


,Employee_ID,Task_ID,Assigned_Hours,Cost_Generated
0,E04,P01_Backend,500.0,30500.0
1,E06,P01_Frontend,600.0,31200.0
2,E05,P01_QA,300.0,11100.0
3,E04,P02_Backend,900.0,54900.0
4,E05,P02_QA,400.0,14800.0
5,E04,P03_Backend,500.0,30500.0
6,E06,P03_Frontend,200.0,10400.0
7,E05,P03_QA,200.0,7400.0
8,E04,P04_Backend,180.0,10980.0
9,E08,P04_Backend,320.0,23040.0



--- Employee Utilization Check ---


,Employee_ID,Assigned_Hours,Max_Hours,Utilization_%
0,E04,2080.0,2080,100.0
1,E05,2000.0,2080,96.2
2,E06,1900.0,2080,91.3
3,E08,820.0,2080,39.4


# Option 2: Multiple skills per employee - one skill per task

In [ ]:
import pandas as pd
import numpy as np
import random
from ortools.linear_solver import pywraplp

# Set seed for reproducibility
random.seed(42)
np.random.seed(42)

skills = ['Backend', 'Frontend', 'QA', 'DevOps']

# 1. Generate Multi-Skilled Employees
employee_data = []
for i in range(1, 100):  # 14 employees to ensure sufficient capacity
    emp_id = f"E{i:02d}"
    
    # Give each employee between 1 and 3 distinct skills
    emp_skills = random.sample(skills, k=random.randint(1, 3))
    
    # We make "Full-Stack" employees slightly more expensive!
    base_cost = 50 + (len(emp_skills) * 15) 
    cost = base_cost + random.randint(-5, 15) 
    
    employee_data.append({
        'Employee_ID': emp_id,
        'Skills': emp_skills,  # This is now a LIST
        'Hourly_Cost': cost,
        'Max_Hours': 2080
    })


employees_df = pd.DataFrame(employee_data)

# 2. Generate Tasks
task_data = []
for p in range(1, 6): # 5 Projects
    proj_id = f"P{p:02d}"
    roles_needed = random.sample(skills, k=random.randint(2, 4)) 
    
    for role in roles_needed:
        task_id = f"{proj_id}_{role}"
        hours = random.randint(2, 6) * 100 
        task_data.append({
            'Task_ID': task_id,
            'Required_Skill': role,
            'Hours_Needed': hours
        })

tasks_df = pd.DataFrame(task_data)

print("--- Multi-Skilled Employees ---")
display(employees_df.head(10))

# 3. The Magic Update: Generating Valid Pairs
valid_pairs = []
for _, emp in employees_df.iterrows():
    for _, task in tasks_df.iterrows():
        # UPDATE: Check if the task's skill is IN the employee's list of skills
        if task['Required_Skill'] in emp['Skills']:
            valid_pairs.append((emp['Employee_ID'], task['Task_ID']))

--- Multi-Skilled Employees ---


,Employee_ID,Skills,Hourly_Cost,Max_Hours
0,E01,"[Backend, DevOps, Frontend]",97,2080
1,E02,[Frontend],63,2080
2,E03,"[Backend, QA, Frontend]",91,2080
3,E04,[Backend],66,2080
4,E05,[Backend],77,2080
5,E06,[DevOps],67,2080
6,E07,"[QA, Backend]",80,2080
7,E08,"[DevOps, Frontend, QA]",94,2080
8,E09,[QA],63,2080
9,E10,[DevOps],63,2080


In [13]:
# Initialize the OR-Tools solver using GLOP
solver = pywraplp.Solver.CreateSolver('GLOP')

# Define Decision Variables: x_ij
x = {}
for (i, j) in valid_pairs:
    x[(i, j)] = solver.NumVar(0, solver.infinity(), f'x_{i}_{j}')

# Define Objective Function: Minimize Cost
cost_dict = employees_df.set_index('Employee_ID')['Hourly_Cost'].to_dict()
objective = solver.Objective()
for (i, j) in valid_pairs:
    objective.SetCoefficient(x[(i, j)], float(cost_dict[i]))
objective.SetMinimization()

# Constraint 1: Demand Satisfaction 
demand_dict = tasks_df.set_index('Task_ID')['Hours_Needed'].to_dict()
for j in tasks_df['Task_ID']:
    capable_employees = [i for (i, t_id) in valid_pairs if t_id == j]
    constraint = solver.Constraint(float(demand_dict[j]), float(demand_dict[j]), f"Demand_{j}")
    for i in capable_employees:
        constraint.SetCoefficient(x[(i, j)], 1)

# Constraint 2: Capacity Limit 
cap_dict = employees_df.set_index('Employee_ID')['Max_Hours'].to_dict()
for i in employees_df['Employee_ID']:
    assigned_tasks = [j for (e_id, j) in valid_pairs if e_id == i]
    constraint = solver.Constraint(0, float(cap_dict[i]), f"Capacity_{i}")
    for j in assigned_tasks:
        constraint.SetCoefficient(x[(i, j)], 1)

# Solve the model
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print(f"Status: OPTIMAL")
    print(f"Total Optimized Cost: ${solver.Objective().Value():,.2f}")
    
    # Extract results
    results = []
    for (i, j) in valid_pairs:
        assigned_hours = x[(i, j)].solution_value()
        if assigned_hours > 0: 
            results.append({
                'Employee_ID': i,
                'Task_ID': j,
                'Assigned_Hours': assigned_hours
            })
    display(pd.DataFrame(results).sort_values(by=['Task_ID']).reset_index(drop=True))
else:
    print("Infeasible! You need to hire more employees or reduce project scope.")

Status: OPTIMAL
Total Optimized Cost: $266,800.00


,Employee_ID,Task_ID,Assigned_Hours
0,E15,P01_Backend,300.0
1,E23,P01_Frontend,200.0
2,E49,P01_QA,300.0
3,E15,P02_Backend,300.0
4,E36,P02_DevOps,300.0
5,E49,P02_QA,300.0
6,E15,P03_Backend,500.0
7,E36,P03_DevOps,300.0
8,E23,P03_Frontend,400.0
9,E15,P04_Backend,200.0


# Option 3: Multiple skills per employee - multiple skills per task

## 1. Problem Definition
In modern IT environments, tasks rarely require just one isolated skill. A backend task might require both `Python` and `AWS` simultaneously. The worker assigned to this task must possess **all** required skills to be eligible. We call this "Conjunctive Skill Matching."

Furthermore, to prevent the model from crashing when our internal workforce lacks a specific combination of skills (the "Unicorn Shortage"), we introduce an **Artificial Variable** in the form of an infinitely capable, highly expensive "External Contractor."

## 2. Mathematical Formulation
**Sets:**
* $I$: Set of all employees.
* $J$: Set of all tasks.
* $S_i$: The set of skills possessed by employee $i$.
* $R_j$: The set of skills required by task $j$.
* $V$: The set of valid assignment pairs. A pair $(i, j) \in V$ if and only if $R_j \subseteq S_i$ (the task's required skills are a subset of the employee's skills).

**Parameters:**
* $C_i$: Hourly cost of employee $i$ (Contractor cost $C_{ext} \gg$ Internal cost).
* $K_i$: Maximum annual available hours for employee $i$.
* $D_j$: Total hours demanded by task $j$.

**Decision Variables:**
* $x_{ij} \ge 0$: The continuous number of hours employee $i$ is assigned to task $j$. Defined only for $(i,j) \in V$.

**Objective Function:**
Minimize total execution cost:
$$\min Z = \sum_{(i,j) \in V} C_i \cdot x_{ij}$$

**Constraints:**
1. **Demand Satisfaction:** Every task must be fulfilled.
$$\sum_{i \mid (i,j) \in V} x_{ij} = D_j \quad \forall j \in J$$

2. **Capacity Limit:** No employee can exceed their max hours.
$$\sum_{j \mid (i,j) \in V} x_{ij} \le K_i \quad \forall i \in I$$

In [ ]:
import pandas as pd
import numpy as np
import random
from ortools.linear_solver import pywraplp

# Set seed for reproducible presentations
random.seed(10)
np.random.seed(10)

tech_stack = ['Python', 'JavaScript', 'SQL', 'AWS', 'Docker']

# 1. Generate Internal Employees
employee_data = []
for i in range(1, 50):  # 15 internal employees
    emp_id = f"E{i:02d}"
    
    # Give each employee 1 to 3 skills
    emp_skills = random.sample(tech_stack, k=random.randint(1, 3))
    
    # Base cost scales with the number of skills they know
    cost = 50 + (len(emp_skills) * 15) + random.randint(-5, 10)
    
    employee_data.append({
        'Employee_ID': emp_id,
        'Skills': emp_skills,
        'Hourly_Cost': cost,
        'Max_Hours': 2080
    })

# THE OR TRICK: Add the External Contractor to guarantee feasibility
employee_data.append({
    'Employee_ID': 'EXT_CONTRACTOR',
    'Skills': tech_stack,  # Possesses EVERY skill
    'Hourly_Cost': 5000,   # Exorbitantly expensive
    'Max_Hours': 999999    # Effectively infinite capacity
})

employees_df = pd.DataFrame(employee_data)

# 2. Generate Complex Tasks
task_data = []
for p in range(1, 6): # 5 Projects
    proj_id = f"P{p:02d}"
    
    # 2 tasks per project
    for t in range(1, 3):
        task_id = f"{proj_id}_Task{t}"
        # Task requires a combination of 1 to 3 skills
        req_skills = random.sample(tech_stack, k=random.randint(1, 3))
        hours = random.randint(3, 8) * 100 
        
        task_data.append({
            'Task_ID': task_id, #
            'Required_Skills': req_skills,
            'Hours_Needed': hours
        })

tasks_df = pd.DataFrame(task_data)

print("--- Internal Workforce + Contractor ---")
display(pd.concat([employees_df.head(3), employees_df.tail(2)])) # Show top internal and the contractor

print("\n--- Project Tasks (Demand) ---")
display(tasks_df.head(5))

--- Internal Workforce + Contractor ---


,Employee_ID,Skills,Hourly_Cost,Max_Hours
0,E01,"[Python, AWS, JavaScript]",90,2080
1,E02,[AWS],75,2080
2,E03,"[JavaScript, Python]",90,2080
48,E49,[Python],66,2080
49,EXT_CONTRACTOR,"[Python, JavaScript, SQL, AWS, Docker]",5000,999999



--- Project Tasks (Demand) ---


,Task_ID,Required_Skills,Hours_Needed
0,P01_Task1,"[Docker, AWS]",600
1,P01_Task2,"[JavaScript, Python, Docker]",300
2,P02_Task1,[SQL],500
3,P02_Task2,"[AWS, Docker]",400
4,P03_Task1,"[SQL, AWS, Docker]",500


In [22]:
# 3. Generate Valid Pairs using Set Theory
valid_pairs = []
for _, emp in employees_df.iterrows():
    emp_skill_set = set(emp['Skills']) 
    for _, task in tasks_df.iterrows():
        task_req_set = set(task['Required_Skills'])
        # subset check: employee must have ALL skills required by the task
        if task_req_set.issubset(emp_skill_set):
            valid_pairs.append((emp['Employee_ID'], task['Task_ID']))

print(f"Generated {len(valid_pairs)} valid assignment combinations.\n")

# 4. Initialize OR-Tools Solver
solver = pywraplp.Solver.CreateSolver('GLOP')

# Decision Variables
x = {}
for (i, j) in valid_pairs:
    x[(i, j)] = solver.NumVar(0, solver.infinity(), f'x_{i}_{j}')

# Objective Function
cost_dict = employees_df.set_index('Employee_ID')['Hourly_Cost'].to_dict()
objective = solver.Objective()
for (i, j) in valid_pairs:
    objective.SetCoefficient(x[(i, j)], float(cost_dict[i]))
objective.SetMinimization()

# Constraint 1: Demand
demand_dict = tasks_df.set_index('Task_ID')['Hours_Needed'].to_dict()
for j in tasks_df['Task_ID']:
    capable_employees = [i for (i, t_id) in valid_pairs if t_id == j]
    constraint = solver.Constraint(float(demand_dict[j]), float(demand_dict[j]), f"Demand_{j}")
    for i in capable_employees:
        constraint.SetCoefficient(x[(i, j)], 1)

# Constraint 2: Capacity
cap_dict = employees_df.set_index('Employee_ID')['Max_Hours'].to_dict()
for i in employees_df['Employee_ID']:
    assigned_tasks = [j for (e_id, j) in valid_pairs if e_id == i]
    constraint = solver.Constraint(0, float(cap_dict[i]), f"Capacity_{i}")
    for j in assigned_tasks:
        constraint.SetCoefficient(x[(i, j)], 1)

# 5. Solve
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print(f"Status: OPTIMAL")
    print(f"Total Portfolio Cost: ${solver.Objective().Value():,.2f}")
    
    # Extract results
    results = []
    for (i, j) in valid_pairs:
        assigned_hours = x[(i, j)].solution_value()
        if assigned_hours > 0: 
            results.append({
                'Task_ID': j,
                'Employee_ID': i,
                'Assigned_Hours': assigned_hours,
                'Task_Cost': assigned_hours * cost_dict[i]
            })
    
    results_df = pd.DataFrame(results).sort_values(by=['Task_ID', 'Employee_ID']).reset_index(drop=True)
    
    print("\n--- Final Assignment Schedule ---")
    display(results_df)
    
    # Analytics for Presentation: Did we use the contractor?
    contractor_hours = results_df[results_df['Employee_ID'] == 'EXT_CONTRACTOR']['Assigned_Hours'].sum()
    if contractor_hours > 0:
        print(f"\n🚨 BUSINESS ALERT: Internal workforce lacked capacity/skills.")
        print(f"🚨 Forced to outsource {contractor_hours:,.0f} hours to the External Contractor.")
        display(results_df[results_df['Employee_ID'] == 'EXT_CONTRACTOR'])
    else:
        print("\n✅ SUCCESS: All projects staffed purely by internal workforce.")

else:
    print("Solver failed. Status:", status)

Generated 77 valid assignment combinations.

Status: OPTIMAL
Total Portfolio Cost: $472,800.00

--- Final Assignment Schedule ---


,Task_ID,Employee_ID,Assigned_Hours,Task_Cost
0,P01_Task1,E09,600.0,45600.0
1,P01_Task2,E34,300.0,27600.0
2,P02_Task1,E22,500.0,37000.0
3,P02_Task2,E09,400.0,30400.0
4,P03_Task1,E37,500.0,49000.0
5,P03_Task2,E39,600.0,54600.0
6,P04_Task1,E37,500.0,49000.0
7,P04_Task2,E09,700.0,53200.0
8,P05_Task1,E13,800.0,64000.0
9,P05_Task2,E32,800.0,62400.0



✅ SUCCESS: All projects staffed purely by internal workforce.
